In [ ]:
# Malariagen API arguments for each call below.
# Replace a value to override it; keep the marker to use the call-site
# value or the method's default.
API_KEEP_CALLSITE_VALUE = '__KEEP_CALLSITE_VALUE__'
PERFORMANCE_CSV_PATH = '../executed_runs/plot_biallelic_snps_performance.csv'
API_CALL_PARAMETERS = {'cell_6_call_1_ag3_biallelic_snp_calls_ld_pruned': {'region': '3L',
                                                     'n_snps': 100000,
                                                     'ld_window_size': 500,
                                                     'ld_window_step': 200,
                                                     'ld_threshold': 0.1,
                                                     'thin_offset': 0,
                                                     'sample_sets': 'AG1000G-BF-A',
                                                     'sample_query': None,
                                                     'sample_query_options': None,
                                                     'sample_indices': None,
                                                     'site_mask': 'default',
                                                     'min_minor_ac': 2,
                                                     'max_missing_an': 0,
                                                     'random_seed': 42,
                                                     'inline_array': True,
                                                     'chunks': 'native'},
 'cell_9_call_1_ag3_biallelic_snp_calls_ld_pruned': {'region': '3L',
                                                     'n_snps': 100000,
                                                     'ld_window_size': 500,
                                                     'ld_window_step': 200,
                                                     'ld_threshold': 0.1,
                                                     'thin_offset': 0,
                                                     'sample_sets': 'AG1000G-BF-A',
                                                     'sample_query': None,
                                                     'sample_query_options': None,
                                                     'sample_indices': None,
                                                     'site_mask': 'default',
                                                     'min_minor_ac': 2,
                                                     'max_missing_an': 0,
                                                     'random_seed': 42,
                                                     'inline_array': True,
                                                     'chunks': 'native'},
 'cell_9_call_2_ag3_biallelic_snp_calls_ld_pruned': {'region': '3L',
                                                     'n_snps': 100000,
                                                     'ld_window_size': 500,
                                                     'ld_window_step': 200,
                                                     'ld_threshold': 0.5,
                                                     'thin_offset': 0,
                                                     'sample_sets': 'AG1000G-BF-A',
                                                     'sample_query': None,
                                                     'sample_query_options': None,
                                                     'sample_indices': None,
                                                     'site_mask': 'default',
                                                     'min_minor_ac': 2,
                                                     'max_missing_an': 0,
                                                     'random_seed': 42,
                                                     'inline_array': True,
                                                     'chunks': 'native'},
 'cell_11_call_1_ag3_biallelic_snp_calls': {'region': '3L',
                                            'sample_sets': 'AG1000G-BF-A',
                                            'sample_query': None,
                                            'sample_query_options': None,
                                            'sample_indices': None,
                                            'site_mask': None,
                                            'site_class': None,
                                            'inline_array': True,
                                            'chunks': 'native',
                                            'cohort_size': None,
                                            'min_cohort_size': None,
                                            'max_cohort_size': None,
                                            'random_seed': 42,
                                            'min_minor_ac': None,
                                            'max_missing_an': None,
                                            'n_snps': 100000,
                                            'thin_offset': 0}}


In [ ]:
import inspect
from memory_profiler import memory_usage
import time
import json
import csv
from pathlib import Path

def _parameter_value_for_csv(value):
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, dict):
        return {str(key): _parameter_value_for_csv(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_parameter_value_for_csv(item) for item in value]
    shape = getattr(value, "shape", None)
    if shape is not None:
        return {
            "type": f"{type(value).__module__}.{type(value).__qualname__}",
            "shape": list(shape),
        }
    return repr(value)


def _call_api_with_parameters(method, call_key, *args, **kwargs):
    signature = inspect.signature(method)
    bound = signature.bind_partial(*args, **kwargs)
    bound.apply_defaults()
    for name, value in API_CALL_PARAMETERS[call_key].items():
        if not (isinstance(value, str) and value == API_KEEP_CALLSITE_VALUE):
            bound.arguments[name] = value

    positional = []
    keyword = {}
    for parameter in signature.parameters.values():
        if parameter.kind is inspect.Parameter.POSITIONAL_ONLY:
            if parameter.name in bound.arguments:
                positional.append(bound.arguments[parameter.name])
        elif parameter.kind is inspect.Parameter.POSITIONAL_OR_KEYWORD:
            if parameter.name in bound.arguments:
                keyword[parameter.name] = bound.arguments[parameter.name]
        elif parameter.kind is inspect.Parameter.VAR_POSITIONAL:
            positional.extend(bound.arguments.get(parameter.name, ()))
        elif parameter.kind is inspect.Parameter.KEYWORD_ONLY:
            if parameter.name in bound.arguments:
                keyword[parameter.name] = bound.arguments[parameter.name]
        elif parameter.kind is inspect.Parameter.VAR_KEYWORD:
            keyword.update(bound.arguments.get(parameter.name, {}))
    result = {}

    def operation():
        result["value"] = method(*positional, **keyword)

    start_time = time.perf_counter()
    peak_memory_mib = memory_usage((operation, ()), max_usage=True)
    runtime_sec = time.perf_counter() - start_time

    output_path = Path(PERFORMANCE_CSV_PATH)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = [
        "call_key",
        "method",
        "parameters_json",
        "runtime_sec",
        "peak_memory_mib",
    ]
    record = {
        "call_key": call_key,
        "method": getattr(method, "__qualname__", getattr(method, "__name__", repr(method))),
        "parameters_json": json.dumps(
            {
                name: _parameter_value_for_csv(value)
                for name, value in bound.arguments.items()
            },
            sort_keys=True,
        ),
        "runtime_sec": round(runtime_sec, 4),
        "peak_memory_mib": round(peak_memory_mib, 2),
    }
    file_exists = output_path.exists() and output_path.stat().st_size > 0
    with output_path.open("a", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        writer.writerow(record)

    return result["value"]


# LD-pruned biallelic SNP calls

LD pruning removes redundant SNPs that are in linkage disequilibrium, reducing correlation between variants. This is important for analyses like PCA and ADMIXTURE where independent markers are assumed.

`biallelic_snp_calls_ld_pruned()` wraps `biallelic_snp_calls()` and applies LD pruning via `scikit-allel`'s `locate_unlinked()`. The output retains the same dataset structure and is compatible with existing downstream methods.

## Setup

In [ ]:
import malariagen_data

In [ ]:
ag3 = malariagen_data.Ag3(
    "simplecache::gs://vo_agam_release_master_us_central1",
    simplecache=dict(cache_storage="../gcs_cache"),
    results_cache="results_cache",
)
ag3

## Basic usage

In [ ]:
ds_pruned = _call_api_with_parameters(ag3.biallelic_snp_calls_ld_pruned, 'cell_6_call_1_ag3_biallelic_snp_calls_ld_pruned', region="3L", n_snps=100_000, sample_sets="AG1000G-BF-A")
ds_pruned

In [ ]:
# Inspect dimensions.
print(f"variants: {ds_pruned.sizes['variants']}")
print(f"samples:  {ds_pruned.sizes['samples']}")
print(f"alleles:  {ds_pruned.sizes['alleles']}")

## Effect of LD parameters

In [ ]:
# Stricter threshold (0.1, default) removes more correlated SNPs.
ds_strict = _call_api_with_parameters(ag3.biallelic_snp_calls_ld_pruned, 'cell_9_call_1_ag3_biallelic_snp_calls_ld_pruned', region="3L", n_snps=100_000, sample_sets="AG1000G-BF-A", ld_threshold=0.1)

# More lenient threshold retains more SNPs.
ds_lenient = _call_api_with_parameters(ag3.biallelic_snp_calls_ld_pruned, 'cell_9_call_2_ag3_biallelic_snp_calls_ld_pruned', region="3L", n_snps=100_000, sample_sets="AG1000G-BF-A", ld_threshold=0.5)

print(f"strict  (threshold=0.1): {ds_strict.sizes['variants']} variants")
print(f"lenient (threshold=0.5): {ds_lenient.sizes['variants']} variants")

## Before vs after pruning

In [ ]:
# Get the thinned (but not LD-pruned) SNPs for comparison.
ds_before = _call_api_with_parameters(ag3.biallelic_snp_calls, 'cell_11_call_1_ag3_biallelic_snp_calls', region="3L", n_snps=100_000, sample_sets="AG1000G-BF-A")

print(f"before LD pruning: {ds_before.sizes['variants']} variants")
print(f"after  LD pruning: {ds_pruned.sizes['variants']} variants")
print(f"removed:           {ds_before.sizes['variants'] - ds_pruned.sizes['variants']} variants")

## Downstream compatibility

The pruned dataset retains the same structure as `biallelic_snp_calls()` output, so it can be passed directly into existing workflows like PCA.

In [ ]:
# Verify the pruned dataset has all variables expected by downstream methods.
assert "call_genotype" in ds_pruned
assert "variant_allele" in ds_pruned
assert "variant_contig" in ds_pruned.coords
assert "variant_position" in ds_pruned.coords
assert "sample_id" in ds_pruned.coords

# Shape sanity check.
n_variants = ds_pruned.sizes["variants"]
n_samples = ds_pruned.sizes["samples"]
assert ds_pruned["call_genotype"].shape == (n_variants, n_samples, 2)
assert ds_pruned["variant_allele"].shape == (n_variants, 2)

print(f"Dataset is valid: {n_variants} variants × {n_samples} samples")